In [48]:
import torch
import numpy as np

In [31]:
## example 1
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [32]:
y = x**2

In [33]:
y

tensor(9., grad_fn=<PowBackward0>)

In [34]:
y.backward() ## backward gradinents

In [35]:
x.grad

tensor(6.)

In [36]:
## example 2
# y = x**2
# z = sin(y)
# dz/dx??
# dz/dx = dz/dy . dy/dx
#       = cos(y) . 2x  = 2 x cos(x**2) 

x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = torch.sin(y)
x,y,z

(tensor(3., requires_grad=True),
 tensor(9., grad_fn=<PowBackward0>),
 tensor(0.4121, grad_fn=<SinBackward0>))

In [37]:
z.backward()

In [39]:
x.grad

tensor(-5.4668)

In [50]:
2*3*np.cos(9)

np.float64(-5.466781571308061)

In [53]:
y.grad

/var/folders/c5/5xrh1cx90sq0syx0xry3fzqr0000gn/T/ipykernel_22360/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


## Logistic Loss Example

### Core Functions

**1. Linear Transformation**
$$z = w x + b$$

**2. Sigmoid Activation** 
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**3. Binary Cross-Entropy Loss**
$$\mathcal{L} = -[y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

### Gradient Derivation

$$\frac{\partial\mathcal{L}}{\partial z} = \hat{y} - y$$

$$\frac{\partial\mathcal{L}}{\partial w} = \frac{\partial\mathcal{L}}{\partial z} \cdot \frac{\partial z}{\partial w} = (\hat{y} - y) \cdot x$$

$$\frac{\partial\mathcal{L}}{\partial b} = \frac{\partial\mathcal{L}}{\partial z} \cdot \frac{\partial z}{\partial b} = \hat{y} - y$$


### Detailed Derivation: $\frac{\partial\mathcal{L}}{\partial z} = \hat{y} - y$

Start with Binary Cross-Entropy:
$$\mathcal{L} = -[y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

where $\hat{y} = \sigma(z) = \frac{1}{1+e^{-z}}$

**Step 1:** Differentiate $\mathcal{L}$ w.r.t. $\hat{y}$
$$\frac{\partial\mathcal{L}}{\partial\hat{y}} = -\left[ \frac{y}{\hat{y}} - \frac{1-y}{1-\hat{y}} \right] = \frac{\hat{y}-y}{\hat{y}(1-\hat{y})}$$

**Step 2:** Sigmoid derivative $\frac{\partial\hat{y}}{\partial z}$
$$\frac{\partial\hat{y}}{\partial z} = \hat{y}(1-\hat{y})$$

**Step 3:** Chain Rule $\frac{\partial\mathcal{L}}{\partial z}$
$$\frac{\partial\mathcal{L}}{\partial z} = \frac{\partial\mathcal{L}}{\partial\hat{y}} \cdot \frac{\partial\hat{y}}{\partial z}$$

Substitute both:
$$\frac{\partial\mathcal{L}}{\partial z} = \left(-\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}\right) \cdot \hat{y}(1-\hat{y})$$

**Step 4:** Simplify algebraically

First term: $-\frac{y}{\hat{y}} \cdot \hat{y}(1-\hat{y}) = -y(1-\hat{y})$

Second term: $\frac{1-y}{1-\hat{y}} \cdot \hat{y}(1-\hat{y}) = (1-y)\hat{y}$

Total: $-y + y\hat{y} + \hat{y} - y\hat{y} = \hat{y} - y$


In [61]:
# Inputs
x = torch.tensor(6.7)  # Input feature (say cgpa)
y = torch.tensor(0.0)  # True label (binary) (placement >> NO)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias


# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon) ##torch.clamp(tensor, min=?, max=?)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))


# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [62]:
loss

tensor(6.7012)

#### Derivatives manualy

In [78]:
# dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [60]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


### derivative wrt AutoGrad

In [79]:
x = torch.tensor(6.7) 
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True) ## requir grads wrt w and b
b = torch.tensor(0.0, requires_grad=True)

w,b

(tensor(1., requires_grad=True), tensor(0., requires_grad=True))

In [80]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [81]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [82]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [83]:
loss.backward()

In [84]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)
